# AsyncIO and aiohttp (with Trading/REST Examples)

This notebook is a **from-scratch to intermediate/advanced** guide to:

- **`asyncio`**: Python’s built-in asynchronous I/O framework
- **`aiohttp`**: an async HTTP client/server library

We focus on **concurrency and parallelizing tasks that take an indeterminate amount
of time**, especially **REST API calls** and combining async I/O with **CPU-bound
work** using executors.

### Contents

- [1. Core Ideas: Event Loop, Tasks, and Awaitables](#1-core-ideas-event-loop-tasks-and-awaitables)
- [2. Basic asyncio Patterns (sleep, gather)](#2-basic-asyncio-patterns-sleep-gather)
- [3. Intro to aiohttp Client](#3-intro-to-aiohttp-client)
- [4. Study Case: Parallel Crypto/Market REST Calls](#4-study-case-parallel-cryptomarket-rest-calls)
- [5. Timeouts, Cancellation, and Error Handling](#5-timeouts-cancellation-and-error-handling)
- [6. Combining AsyncIO with CPU-Bound Work (Executors)](#6-combining-asyncio-with-cpu-bound-work-executors)
- [7. Concurrency Limits and Backpressure](#7-concurrency-limits-and-backpressure)
- [8. Best Practices and Pitfalls](#8-best-practices-and-pitfalls)

## 1. Core Ideas: Event Loop, Tasks, and Awaitables

Key concepts:

- **Event loop**: runs asynchronous tasks and callbacks; there is usually one loop per
  OS thread.
- **Coroutine**: an `async def` function that returns an awaitable when called.
- **Task**: a scheduled coroutine managed by the event loop; created with
  `asyncio.create_task` or `asyncio.gather`.
- **Awaitable**: something you can `await` on (coroutine, `asyncio.Task`, `Future`).

AsyncIO provides **concurrency** by interleaving tasks when they hit an `await` on
I/O or other awaitables. It does *not* run Python bytecode in parallel on multiple
cores by itself; for CPU-bound work you combine it with **executors** (thread or
process pools).

In [1]:
# 1.1 Minimal asyncio example: concurrent sleeps

import asyncio
import time


async def sleepy(name: str, delay: float) -> str:
    print(f"{name} sleeping for {delay}s")
    await asyncio.sleep(delay)
    print(f"{name} woke up")
    return name


async def main_basic():
    t0 = time.perf_counter()
    # Schedule three coroutines concurrently
    results = await asyncio.gather(
        sleepy("A", 1.0),
        sleepy("B", 1.5),
        sleepy("C", 0.5),
    )
    elapsed = time.perf_counter() - t0
    print("Results:", results)
    print(f"Total elapsed: {elapsed:.2f}s (max delay ~1.5s)")


# In a notebook, use asyncio.run only once per process; here it's fine for demo
await main_basic()

A sleeping for 1.0s
B sleeping for 1.5s
C sleeping for 0.5s
C woke up
A woke up
B woke up
Results: ['A', 'B', 'C']
Total elapsed: 1.51s (max delay ~1.5s)


## 2. Basic asyncio Patterns (sleep, gather)

Patterns you will use all the time:

- **`asyncio.sleep`**: non-blocking sleep that yields control to the event loop.
- **`asyncio.gather`**: run multiple awaitables concurrently and wait for all.
- **`asyncio.create_task`**: fire-and-forget style scheduling; useful for background
  tasks (e.g. heartbeats, metrics).

We’ll show `gather` for structured concurrency; later sections will use it for
parallel HTTP requests.

In [2]:
# 2.1 Using create_task for background work

async def heartbeat(interval: float = 1.0):
    while True:
        await asyncio.sleep(interval)
        print("[heartbeat] still alive")


async def main_with_heartbeat():
    hb_task = asyncio.create_task(heartbeat(0.7))  # runs until we cancel it
    await asyncio.sleep(2.5)  # simulate other work
    hb_task.cancel()
    try:
        await hb_task
    except asyncio.CancelledError:
        print("Heartbeat cancelled")


await main_with_heartbeat()

[heartbeat] still alive
[heartbeat] still alive
[heartbeat] still alive
Heartbeat cancelled


## 3. Intro to aiohttp Client

`aiohttp` is an async HTTP client/server library built on top of `asyncio`.
We’ll only use the **client** side here.

Core API for GET requests:

```python
import aiohttp

async with aiohttp.ClientSession() as session:
    async with session.get(url, params={...}, timeout=10) as resp:
        data = await resp.json()  # or resp.text(), resp.read()
```

Important points:

- Use a **single `ClientSession`** per process or component and reuse it.
- Use **`async with`** to ensure connections are cleaned up.
- Always **`await`** the response methods (`json()`, `text()`, `read()`).

In [3]:
# 3.1 Simple aiohttp GET to a public crypto endpoint

import aiohttp

# We'll use Binance's public REST API for crypto prices (no auth required).
BINANCE_BASE = "https://api.binance.com"


async def fetch_binance_price(symbol: str, session: aiohttp.ClientSession) -> float | None:
    """Fetch the latest price for a symbol like 'BTCUSDT' from Binance.

    Returns the price as float, or None on error.
    """
    url = f"{BINANCE_BASE}/api/v3/ticker/price"
    params = {"symbol": symbol}
    try:
        async with session.get(url, params=params, timeout=5) as resp:
            resp.raise_for_status()
            data = await resp.json()
            return float(data["price"])
    except Exception as e:
        print(f"Error fetching {symbol}:", e)
        return None


async def main_single_price():
    async with aiohttp.ClientSession() as session:
        price = await fetch_binance_price("BTCUSDT", session)
        print("BTCUSDT price:", price)


await main_single_price()

BTCUSDT price: 71754.78


## 4. Study Case: Parallel Crypto/Market REST Calls

We now build a small **price fetcher** that:

- Queries multiple symbols in parallel using `aiohttp` + `asyncio.gather`.
- Measures total time vs sequential requests.

We’ll use Binance’s public **`/api/v3/ticker/price`** endpoint which requires **no
authentication** and allows multiple symbols.

In [5]:
# 4.1 Sequential vs concurrent price fetches

SYMBOLS = [
    "BTCUSDT",
    "ETHUSDT",
    "BNBUSDT",
    "SOLUSDT",
    "XRPUSDT",
    "DOGEUSDT",
    "ADAUSDT",
    "LTCUSDT",
]


async def fetch_prices_sequential(symbols: list[str]) -> dict[str, float | None]:
    results: dict[str, float | None] = {}
    async with aiohttp.ClientSession() as session:
        for sym in symbols:
            results[sym] = await fetch_binance_price(sym, session)
    return results


async def fetch_prices_concurrent(symbols: list[str]) -> dict[str, float | None]:
    results: dict[str, float | None] = {}
    async with aiohttp.ClientSession() as session:
        tasks = {sym: asyncio.create_task(fetch_binance_price(sym, session)) for sym in symbols}
        for sym, task in tasks.items():
            results[sym] = await task
    return results


async def compare_sequential_vs_concurrent():
    t0 = time.perf_counter()
    seq = await fetch_prices_sequential(SYMBOLS)
    t_seq = time.perf_counter() - t0

    t0 = time.perf_counter()
    conc = await fetch_prices_concurrent(SYMBOLS)
    t_conc = time.perf_counter() - t0

    print("Sequential:")
    print("  elapsed =", f"{t_seq:.2f}s")
    print("Concurrent:")
    print("  elapsed =", f"{t_conc:.2f}s")
    print("Speedup ~", f"{t_seq/t_conc:.1f}x" if t_conc > 0 else "n/a")
    print("Example prices (concurrent):")
    for sym in SYMBOLS:
        print(" ", sym, "->", conc[sym])


await compare_sequential_vs_concurrent()

Sequential:
  elapsed = 2.41s
Concurrent:
  elapsed = 0.40s
Speedup ~ 6.1x
Example prices (concurrent):
  BTCUSDT -> 71747.75
  ETHUSDT -> 2103.0
  BNBUSDT -> 665.85
  SOLUSDT -> 89.13
  XRPUSDT -> 1.4342
  DOGEUSDT -> 0.09749
  ADAUSDT -> 0.2726
  LTCUSDT -> 55.34


## 5. Timeouts, Cancellation, and Error Handling

In real trading/market-data code, calls may **hang** or be **slow**. AsyncIO and
`aiohttp` give you tools to:

- Apply **per-request timeouts**.
- Cancel tasks if they exceed a global timeout.
- Handle partial failures while other tasks complete.

We’ll wrap our fetcher with a timeout and show how to use `asyncio.wait_for` and
`asyncio.gather(..., return_exceptions=True)`.

In [9]:
# 5.1 Per-request timeout and handling exceptions in gather

from aiohttp import ClientTimeout


async def fetch_binance_price_with_timeout(symbol: str, session: aiohttp.ClientSession, timeout_s: float = 3.0) -> float | None:
    url = f"{BINANCE_BASE}/api/v3/ticker/price"
    params = {"symbol": symbol}
    try:
        async with session.get(url, params=params, timeout=ClientTimeout(total=timeout_s)) as resp:
            resp.raise_for_status()
            data = await resp.json()
            return float(data["price"])
    except Exception as e:
        print(f"[{symbol}] error:", e)
        return None


async def fetch_all_with_gather(symbols: list[str]) -> dict[str, float | None]:
    async with aiohttp.ClientSession() as session:
        coros = [fetch_binance_price_with_timeout(sym, session) for sym in symbols]
        results_list = await asyncio.gather(*coros, return_exceptions=True)
    out: dict[str, float | None] = {}
    for sym, res in zip(symbols, results_list):
        if isinstance(res, Exception):
            print(f"{sym} failed with exception:", res)
            out[sym] = None
        else:
            out[sym] = res
    return out


await fetch_all_with_gather(SYMBOLS)

{'BTCUSDT': 71620.59,
 'ETHUSDT': 2097.3,
 'BNBUSDT': 664.5,
 'SOLUSDT': 88.83,
 'XRPUSDT': 1.4312,
 'DOGEUSDT': 0.09725,
 'ADAUSDT': 0.2716,
 'LTCUSDT': 55.25}

In [10]:
# 5.2 Global timeout with asyncio.wait_for

async def fetch_all_with_global_timeout(symbols: list[str], global_timeout: float = 2.0):
    async with aiohttp.ClientSession() as session:
        coros = [fetch_binance_price(sym, session) for sym in symbols]
        try:
            t0 = time.perf_counter()
            results = await asyncio.wait_for(asyncio.gather(*coros), timeout=global_timeout)
            elapsed = time.perf_counter() - t0
            print(f"All done within global timeout {global_timeout}s, elapsed={elapsed:.2f}s")
            return dict(zip(symbols, results))
        except asyncio.TimeoutError:
            elapsed = time.perf_counter() - t0
            print(f"Global timeout {global_timeout}s hit after {elapsed:.2f}s; tasks cancelled")
            return {}


await fetch_all_with_global_timeout(SYMBOLS, global_timeout=1.0)

All done within global timeout 1.0s, elapsed=0.42s


{'BTCUSDT': 71604.56,
 'ETHUSDT': 2096.35,
 'BNBUSDT': 664.4,
 'SOLUSDT': 88.8,
 'XRPUSDT': 1.4309,
 'DOGEUSDT': 0.09725,
 'ADAUSDT': 0.2716,
 'LTCUSDT': 55.24}

## 6. Combining AsyncIO with CPU-Bound Work (Executors)

AsyncIO excels at **I/O-bound** concurrency; for **CPU-bound** code (e.g. a heavy
indicator or risk calculation) you still need multiple threads or processes to
use multiple cores.

The typical pattern:

- Run I/O (HTTP, WebSockets, DB) with `asyncio` / `aiohttp`.
- Offload CPU-bound functions to a **thread pool** or **process pool** using
  `loop.run_in_executor` or `asyncio.to_thread`.

Below we simulate fetching prices asynchronously and then running a heavy CPU
calculation on them in parallel using a `ProcessPoolExecutor`.

In [ ]:
# 6.1 CPU-bound function and process pool integration

from concurrent.futures import ProcessPoolExecutor


def cpu_heavy_indicator(prices: list[float]) -> float:
    """Simulate a CPU-heavy indicator calculation (pure Python loop)."""
    total = 0.0
    for p in prices:
        # Some arbitrary math to burn CPU
        total += (p * 0.001) ** 0.5
    return total


async def fetch_prices_for_indicator(symbol: str, session: aiohttp.ClientSession) -> list[float]:
    """Fetch a few recent prices (repeated REST calls) for a symbol."""
    # For demo, fetch the current price N times; in reality you might query a history API
    N = 5
    prices: list[float] = []
    for _ in range(N):
        px = await fetch_binance_price(symbol, session)
        if px is not None:
            prices.append(px)
    return prices


async def run_indicators_with_processpool(symbols: list[str]):
    loop = asyncio.get_running_loop()
    results: dict[str, float] = {}

    async with aiohttp.ClientSession() as session:
        # Step 1: fetch input data concurrently (I/O-bound)
        price_lists = await asyncio.gather(
            *[fetch_prices_for_indicator(sym, session) for sym in symbols]
        )

    # Step 2: run CPU-heavy indicators in a process pool (CPU-bound)
    with ProcessPoolExecutor() as pool:
        tasks = []
        for sym, prices in zip(symbols, price_lists):
            # Offload CPU work to separate process
            tasks.append(loop.run_in_executor(pool, cpu_heavy_indicator, prices))

        indicator_values = await asyncio.gather(*tasks)

    for sym, val in zip(symbols, indicator_values):
        results[sym] = val

    return results


symbols_small = SYMBOLS[:4]
start = time.perf_counter()
vals = await run_indicators_with_processpool(symbols_small)
elapsed = time.perf_counter() - start
print("Indicator values:")
for sym, v in vals.items():
    print(" ", sym, "->", v)
print(f"Total elapsed (I/O+CPU with process pool): {elapsed:.2f}s")

## 7. Concurrency Limits and Backpressure

When calling public APIs (especially crypto exchanges), you must respect **rate limits**.
AsyncIO makes it very easy to fire hundreds of requests at once — too easy.

Typical patterns to **limit concurrency**:

- Use an **`asyncio.Semaphore`** to cap the number of in-flight requests.
- Schedule tasks in **batches** instead of all at once.

We’ll add a semaphore around our `fetch_binance_price` calls to avoid spamming the API.

In [16]:
# 7.1 Limiting concurrency with a semaphore

async def fetch_with_semaphore(symbol: str, session: aiohttp.ClientSession, sem: asyncio.Semaphore) -> float | None:
    async with sem:  # only N coroutines can enter at once
        return await fetch_binance_price(symbol, session)


async def fetch_all_limited(symbols: list[str], max_concurrent: int = 3) -> dict[str, float | None]:
    sem = asyncio.Semaphore(max_concurrent)
    async with aiohttp.ClientSession() as session:
        tasks = [
            asyncio.create_task(fetch_with_semaphore(sym, session, sem))
            for sym in symbols
        ]
        prices = await asyncio.gather(*tasks)
    return dict(zip(symbols, prices))


n = 2
prices_limited = await fetch_all_limited(SYMBOLS, max_concurrent=n)
print(f"Prices with max_concurrent={n}:")
for sym, px in prices_limited.items():
    print(" ", sym, "->", px)

Prices with max_concurrent=2:
  BTCUSDT -> 71777.95
  ETHUSDT -> 2102.86
  BNBUSDT -> 665.37
  SOLUSDT -> 89.08
  XRPUSDT -> 1.4352
  DOGEUSDT -> 0.09766
  ADAUSDT -> 0.2725
  LTCUSDT -> 55.35


## 8. Best Practices and Pitfalls

**Best practices**

- Use **one `ClientSession` per service** and reuse it; creating many sessions is costly.
- Prefer **`asyncio.gather`** for structured concurrency instead of many `create_task`
  without tracking.
- Use **timeouts** (both per-request and global) to avoid hung tasks.
- For CPU-bound work, offload to **executors** (thread/process pools) so the event loop
  stays responsive.
- Implement **concurrency limits** (semaphores) and respect external **rate limits**.

**Pitfalls**

- Mixing **blocking I/O** (e.g. `requests.get`) inside `async def` without offloading:
  this stalls the event loop.
- Calling `asyncio.run` from inside another event loop (e.g. Jupyter); use `await` in
  notebooks or `nest_asyncio` only if you know why.
- Creating too many concurrent tasks at once; use batching or semaphores.
- Forgetting to handle **cancellations** and **exceptions**; `gather(..., return_exceptions=True)`
  can help in fan-out/fan-in patterns.

AsyncIO + aiohttp give you a powerful model for **high-concurrency I/O-bound trading
components** (market data collectors, REST-based execution adapters). Combined with
**executors** for CPU-bound parts, you can keep a single async architecture while
still using multiple cores where it matters.